In [ ]:
# START HERE
!pip install -q implicit threadpoolctl pyarrow scipy pandas numpy scikit-learn mlflow
# Necessary installs for the model

In [25]:
# --- Exploratory Data Analysis: Inspecting the JSON Schema ---
import json
import glob
import os

# Update this path if your data is located elsewhere!
data_folder = '/Users/karthikraturi/Documents/MPD_Playlist_Prediction_Spotify_Project/spotify_million_playlist_dataset/data/'

# Grab just the first file
sample_file = glob.glob(os.path.join(data_folder, "*.json"))[0]

print(f"Inspecting file: {os.path.basename(sample_file)}\n")

with open(sample_file, 'r') as f:
    data = json.load(f)
    
    print("--- Top Level Info ---")
    print(f"Top-level keys: {list(data.keys())}")
    
    # Grab the first playlist
    first_playlist = data['playlists'][0]
    print("\n--- Playlist Level Columns ---")
    for key in first_playlist.keys():
        if key != 'tracks':
            print(f"- {key}: {type(first_playlist[key]).__name__} (i.e.: {first_playlist[key]})")
            
    # Grab the first track of the first playlist
    first_track = first_playlist['tracks'][0]
    print("\n--- Track Level Columns ---")
    for key in first_track.keys():
        print(f"- {key}: {type(first_track[key]).__name__} (i.e.: {first_track[key]})")

Inspecting file: mpd.slice.549000-549999.json

--- Top Level Info ---
Top-level keys: ['info', 'playlists']

--- Playlist Level Columns ---
- name: str (i.e.: Bob Dylan)
- collaborative: str (i.e.: false)
- pid: int (i.e.: 549000)
- modified_at: int (i.e.: 1454803200)
- num_tracks: int (i.e.: 75)
- num_albums: int (i.e.: 65)
- num_followers: int (i.e.: 1)
- num_edits: int (i.e.: 28)
- duration_ms: int (i.e.: 18425368)
- num_artists: int (i.e.: 39)

--- Track Level Columns ---
- pos: int (i.e.: 0)
- artist_name: str (i.e.: Bob Dylan)
- track_uri: str (i.e.: spotify:track:6QHYEZlm9wyfXfEM1vSu1P)
- artist_uri: str (i.e.: spotify:artist:74ASZWbe4lXaubB36ztrGX)
- track_name: str (i.e.: Boots of Spanish Leather)
- album_uri: str (i.e.: spotify:album:7DZeLXvr9eTVpyI1OlqtcS)
- duration_ms: int (i.e.: 277106)
- album_name: str (i.e.: The Times They Are A-Changin')


In [29]:
# Module 0: MPD JSON to Parquet ETL Pipeline
import os
import json
import pandas as pd
import glob
import time

def compile_mpd_data(data_folder, max_files=100):
    print(f"Starting ETL: Processing {max_files} JSON files...")
    start_time = time.time()
    
    files = glob.glob(os.path.join(data_folder, "*.json"))
    if max_files:
        files = files[:max_files]
        
    interactions = []
    track_metadata = {}    # Dict to deduplicate tracks
    playlist_metadata = [] # List to store playlist info even with dupe names
    
    # 1. Single Pass Parsing
    for file_path in files:
        with open(file_path, 'r') as f:
            data = json.load(f)
            
            for playlist in data['playlists']:
                pid = playlist['pid']
                
                # Add to Playlist Metadata Bucket (For the UI)
                playlist_metadata.append({
                    'pid': pid,
                    'name': playlist.get('name', 'Unknown Playlist'),
                    'num_tracks': playlist.get('num_tracks', 0),
                    'num_followers': playlist.get('num_followers', 0)
                })
                
                for track in playlist['tracks']:
                    uri = track['track_uri']
                    
                    # Add to Math Bucket
                    interactions.append({'pid': pid, 'track_uri': uri})
                    
                    # Add to Track Metadata Bucket (Deduplicated)
                    if uri not in track_metadata:
                        track_metadata[uri] = {
                            'track_uri': uri,
                            'track_name': track.get('track_name', ''),
                            'artist_name': track.get('artist_name', '')
                        }
                        
    # 2. Build the DataFrames
    print("Building DataFrames...")
    
    # Interactions (Math)
    df_interactions = pd.DataFrame(interactions)
    df_interactions['pid'] = pd.to_numeric(df_interactions['pid'], downcast='integer')
    df_interactions['track_uri'] = df_interactions['track_uri'].astype('category')
    
    # Track Metadata (UI)
    df_track_meta = pd.DataFrame(list(track_metadata.values()))
    
    # Playlist Metadata (UI)
    df_playlist_meta = pd.DataFrame(playlist_metadata)
    df_playlist_meta['pid'] = pd.to_numeric(df_playlist_meta['pid'], downcast='integer')
    
    # 3. Save to Parquet
    print("Compressing and Saving to Parquet...")
    df_interactions.to_parquet('mpd_interactions.parquet', engine='pyarrow', index=False)
    df_track_meta.to_parquet('mpd_track_metadata.parquet', engine='pyarrow', index=False)
    df_playlist_meta.to_parquet('mpd_playlist_metadata.parquet', engine='pyarrow', index=False)
    
    elapsed = time.time() - start_time
    print(f"\n=== ETL COMPLETE ({elapsed:.2f} seconds) ===")
    print(f"Interactions Shape: {df_interactions.shape[0]:,} rows")
    print(f"Unique Tracks Saved: {df_track_meta.shape[0]:,}")
    print(f"Playlists Saved: {df_playlist_meta.shape[0]:,}")
    print("\nGenerated Files:")
    print("1. mpd_interactions.parquet")
    print("2. mpd_track_metadata.parquet")
    print("3. mpd_playlist_metadata.parquet")

# --- RUN THE ETL ---
# Make sure your data_folder path is correct
data_folder = '/Users/karthikraturi/Documents/MPD_Playlist_Prediction_Spotify_Project/spotify_million_playlist_dataset/data/'

# Set max_files to None if you want to run the entire dataset!
compile_mpd_data(data_folder, max_files=1000)

Starting ETL: Processing 1000 JSON files...
Building DataFrames...
Compressing and Saving to Parquet...

=== ETL COMPLETE (364.92 seconds) ===
Interactions Shape: 66,346,428 rows
Unique Tracks Saved: 2,262,292
Playlists Saved: 1,000,000

Generated Files:
1. mpd_interactions.parquet
2. mpd_track_metadata.parquet
3. mpd_playlist_metadata.parquet


In [1]:
import os
import glob
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit
import time
import threadpoolctl
import gc
import joblib
import mlflow
import mlflow.sklearn
import mlflow

In [3]:
# Module 1: Sliced Data Loading and Matrix creation
# --- Force OpenBLAS to use 1 thread dynamically - Addresses compute concerns / GPU OOM error ---
threadpoolctl.threadpool_limits(1, "blas")
# DEV_MODE Configuration for rapid testing

DEV_MODE = True
DEV_SAMPLE_SIZE = 50000  # Loads 5% of the data for fast prototyping


def load_and_build_sparse_matrix(interactions_path, dev_mode=True):
    start_time = time.time()
    
    # Schema-on-Read Filtering (The DataOps Hack)
    if dev_mode:
        print(f"--- DEV MODE: Loading first {DEV_SAMPLE_SIZE:,} playlists ---")
        df = pd.read_parquet(
            interactions_path, 
            columns=['pid', 'track_uri'],
            filters=[('pid', '<', DEV_SAMPLE_SIZE)]
        )
    else:
        print("--- PROD MODE: Loading full 1M dataset ---")
        df = pd.read_parquet(interactions_path, columns=['pid', 'track_uri'])
        
    print(f"Loaded {len(df):,} interactions. Converting to Sparse Matrix...")
    
    # Categorical Encoding (Assign an Integer ID to every Track URI)
    df['track_uri'] = df['track_uri'].astype('category')
    
    # Build the CSR Sparse Matrix
    # rows = playlists, cols = tracks, data = '1' (implicit positive feedback)
    playlists = df['pid'].values
    tracks = df['track_uri'].cat.codes.values
    data = np.ones(len(df), dtype=np.float32) # The implicit library prefers float32
    
    user_item_matrix = sparse.csr_matrix((data, (tracks, playlists)))
    
    # Save the Translation Dictionary
    # We need to map the Integer IDs back to Spotify URIs later
    id_to_uri = dict(enumerate(df['track_uri'].cat.categories))
    uri_to_id = {uri: i for i, uri in id_to_uri.items()}
    
    # Free up RAM by deleting the DataFrame
    del df
    gc.collect()
    
    elapsed = time.time() - start_time
    print(f"Matrix Built! Shape: {user_item_matrix.shape[0]:,} Tracks x {user_item_matrix.shape[1]:,} Playlists")
    print(f"Data Prep Time: {elapsed:.2f} seconds\n")
    
    return user_item_matrix, id_to_uri, uri_to_id

In [5]:
# Module 2: Define Train ALS Engine Function
def train_als_engine(user_item_matrix, factors=100, regularization=0.1, iterations=15):
    print(f"--- Training ALS Engine (Factors: {factors}, Reg: {regularization}, Iterations: {iterations}) ---")
    start_time = time.time()
    
    # Initialize the Alternating Least Squares Model
    # random_state ensures reproducibility for our MLOps tracking
    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42,
        calculate_training_loss=True
    )
    
    # Fit the model using the sparse matrix
    # Under the hood, this uses low-level C++ and all your CPU cores
    model.fit(user_item_matrix)
    
    elapsed = time.time() - start_time
    print(f"Training Complete in {elapsed:.2f} seconds.\n")
    
    return model

In [7]:
# Module 3.1: Popularity Baseline
# Provides a baseline prediction based off of playlist popularity to compare NDCG and MAP@K Scores

import pandas as pd
import numpy as np
from implicit.evaluation import train_test_split
from tqdm.notebook import tqdm

def calculate_ndcg_at_k(recs, truth, k):
    dcg = 0.0
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(truth), k)))
    for i, rec in enumerate(recs[:k]):
        if rec in truth:
            dcg += 1.0 / np.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0.0

def calculate_map_at_k(recs, truth, k):
    hits = 0
    sum_precs = 0.0
    for i, rec in enumerate(recs[:k]):
        if rec in truth:
            hits += 1
            sum_precs += hits / (i + 1.0)
    return sum_precs / min(len(truth), k) if len(truth) > 0 else 0.0

def evaluate_metadata_baseline(track_playlist_matrix, playlist_meta_path, K=10, num_test_tracks=5000):
    print("--- 1. Re-Splitting Data (Matching the ALS Eval) ---")
    train_matrix, test_matrix = train_test_split(track_playlist_matrix, train_percentage=0.8, random_state=42)
    
    print("--- 2. Loading Metadata for True Popularity ---")
    
    # Load the metadata file
    playlist_meta = pd.read_parquet(playlist_meta_path, columns=['pid', 'num_followers'])
    
    # Sort by followers to find the heavy hitters
    top_playlists_df = playlist_meta.sort_values(by='num_followers', ascending=False)
    
    # Extract the PIDs as a clean numpy array for fast iteration
    top_popular_pids = top_playlists_df['pid'].values
    
    print("--- 3. Prepping Test Data ---")
    test_tracks = np.where(test_matrix.getnnz(axis=1) > 0)[0]
    if len(test_tracks) > num_test_tracks:
        np.random.seed(42)
        test_tracks = np.random.choice(test_tracks, num_test_tracks, replace=False)
        
    pop_ndcg_scores, pop_map_scores = [], []
    
    print(f"--- 4. Evaluating Follower Baseline for {num_test_tracks} Tracks ---")
    for track_id in tqdm(test_tracks, desc="Testing Metadata Base"):
        true_playlists = set(test_matrix[track_id].indices)
        if not true_playlists: continue
        train_playlists = set(train_matrix[track_id].indices)
        
        # Build baseline recommendations: top Followed playlists excluding ones the track is already in
        pop_recs = []
        for pid in top_popular_pids:
            if pid not in train_playlists:
                pop_recs.append(pid)
            if len(pop_recs) == K:
                break
                
        # Calculate Metrics
        pop_ndcg_scores.append(calculate_ndcg_at_k(pop_recs, true_playlists, K))
        pop_map_scores.append(calculate_map_at_k(pop_recs, true_playlists, K))
        
    print("\n=== METADATA BASELINE METRICS (BY FOLLOWERS) ===")
    print(f"MAP@{K}:  {np.mean(pop_map_scores):.6f}")
    print(f"NDCG@{K}: {np.mean(pop_ndcg_scores):.6f}\n")
    
    return np.mean(pop_map_scores), np.mean(pop_ndcg_scores)

In [9]:
# Module 3.2: Native Track-to-Playlist Evaluation (MAP@K & NDCG) Function
# Define: Get MAP@K and NDCG Scores for actual model ranking.

from implicit.evaluation import train_test_split, mean_average_precision_at_k, ndcg_at_k
import implicit

def evaluate_als_model(track_playlist_matrix, factors=100, regularization=0.1, iterations=15, K=10):
    print("--- 1. Splitting Data (80% Train / 20% Hidden) ---")
    train_matrix, test_matrix = train_test_split(track_playlist_matrix, train_percentage=0.8, random_state=42)
# This hides 20% of the true playlists for a subset of tracks.

# We MUST train a new model strictly on the 80% split to avoid data leakage.
    eval_model = implicit.als.AlternatingLeastSquares(
        factors=factors, 
        regularization=regularization, 
        iterations=iterations, 
        random_state=42
    )
    eval_model.fit(train_matrix)
    
    print(f"--- 3. Calculating MAP@{K} and NDCG@{K} ---")
    map_score = mean_average_precision_at_k(eval_model, train_matrix, test_matrix, K=K, show_progress=True)
    ndcg_score = ndcg_at_k(eval_model, train_matrix, test_matrix, K=K, show_progress=True)
    
    print("\n=== FINAL EVALUATION METRICS ===")
    print(f" MAP@{K}:  {map_score:.4f}")
    print(f" NDCG@{K}: {ndcg_score:.4f}\n")
    
    return map_score, ndcg_score

In [17]:
# Module 4: Training and Test Execution + MLFlow Logging
    
# --- THE CONTROL PANEL (HYPERPARAMETER TUNING HERE) ---
FACTORS = 64
REGULARIZATION = 0.1
ITERATIONS = 15
K_VAL = 500

# --- Data Pipeline Configuration ---
interactions_file = 'mpd_interactions.parquet'
track_meta_path='mpd_track_metadata.parquet'
playlist_meta_path='mpd_playlist_metadata.parquet'

# --- MLFlow Configuration ---
tracking_path = "file://" + os.path.join(os.getcwd(), "mlruns")
mlflow.set_tracking_uri(tracking_path)
print(f"MLflow is saving to: {mlflow.get_tracking_uri()}")

if mlflow.active_run():
    print("Ending a previous active run...")
    mlflow.end_run()

# --- BUILD SPARSE MATRIX ---
user_items, id_to_uri, uri_to_id = load_and_build_sparse_matrix(
    interactions_file, 
    dev_mode=DEV_MODE
)

# --- RUN THE BASELINE ---
base_map, base_ndcg = evaluate_metadata_baseline(
    track_playlist_matrix=user_items, 
    playlist_meta_path='mpd_playlist_metadata.parquet', # Point to your saved Parquet file
    K=K_VAL,
    num_test_tracks=5000 
)

# --- LOG BASELINE TO MLFLOW ---
with mlflow.start_run(run_name="POPULARITY_BASELINE"):
    mlflow.log_params({
        "K": K_VAL,
        "dev_mode": DEV_MODE,
        "sample_size": DEV_SAMPLE_SIZE if DEV_MODE else 1000000,
        "model_type": "Popularity_Benchmark" # Helps with filtering later
    })
    
    # Log the actual scores from your Module 3.1 results
    mlflow.log_metrics({
        f"MAP_at_{K_VAL}": base_map,
        f"NDCG_at_{K_VAL}": base_ndcg
    })

# --- RUN ALS EXPERIMENT ---
mlflow.set_experiment("ALS_Playlist_Optimizer")

# We wrap both Training and Evaluation in ONE run so they are linked in the UI.
with mlflow.start_run(run_name=f"ALS_Final_f{FACTORS}_r{REGULARIZATION}_iter{ITERATIONS}"):
    mlflow.log_params({
        "factors": FACTORS,
        "regularization": REGULARIZATION,
        "iterations": ITERATIONS,
        "K": K_VAL,
        "dev_mode": DEV_MODE,
        "sample_size": DEV_SAMPLE_SIZE if DEV_MODE else 1000000 
    })
    
    # Train the Model on the Full Dataset (Calling your Module 2 function)
    als_model = train_als_engine(
        user_item_matrix=user_items,
        factors=FACTORS,
        regularization=REGULARIZATION,
        iterations=ITERATIONS
        )
    
    # Evaluate the Model at 80/20 Split
    map_score, ndcg_score = evaluate_als_model(
        track_playlist_matrix=user_items,
        factors=FACTORS,
        regularization=REGULARIZATION,
        iterations=ITERATIONS,
        K=K_VAL
        )   
    # Log the Final Evaluation Results
    mlflow.log_metrics({
        f"MAP_at_{K_VAL}": map_score,
        f"NDCG_at_{K_VAL}": ndcg_score
    })

    # Version Model and Data
    print("Saving latest model to local disk...")
    joblib.dump(uri_to_id, 'uri_to_id.joblib')
    joblib.dump(id_to_uri, 'id_to_uri.joblib')
    joblib.dump(als_model, 'als_model.joblib') 
    sparse.save_npz('user_items.npz', user_items)

    # TELL MLFLOW TO UPLOAD THOSE FILES TO THE DASHBOARD
    mlflow.log_artifact('als_model.joblib', artifact_path="model_artifacts")
    mlflow.log_artifact('uri_to_id.joblib', artifact_path="model_artifacts")

print(f"\n RUN COMPLETE")
print(f"Final Results -> MAP@{K_VAL}: {map_score:.4f} | NDCG@{K_VAL}: {ndcg_score:.4f}")
print(f"Baseline -> MAP@{K_VAL}: {base_map: .6f} | NDCG@{K_VAL}: {base_ndcg:.6f}")

MLflow is saving to: file:///Users/karthikraturi/Documents/MPD_Playlist_Prediction_Spotify_Project/mlruns
--- DEV MODE: Loading first 50,000 playlists ---
Loaded 3,344,374 interactions. Converting to Sparse Matrix...
Matrix Built! Shape: 2,262,283 Tracks x 50,000 Playlists
Data Prep Time: 9.76 seconds

--- 1. Re-Splitting Data (Matching the ALS Eval) ---
--- 2. Loading Metadata for True Popularity ---
--- 3. Prepping Test Data ---
--- 4. Evaluating Follower Baseline for 5000 Tracks ---


Testing Metadata Base:   0%|          | 0/5000 [00:00<?, ?it/s]


=== METADATA BASELINE METRICS (BY FOLLOWERS) ===
MAP@500:  0.000004
NDCG@500: 0.000111

--- Training ALS Engine (Factors: 64, Reg: 0.1, Iterations: 15) ---


  0%|          | 0/15 [00:00<?, ?it/s]

Training Complete in 91.76 seconds.

--- 1. Splitting Data (80% Train / 20% Hidden) ---


  0%|          | 0/15 [00:00<?, ?it/s]

--- 3. Calculating MAP@500 and NDCG@500 ---


  0%|          | 0/173504 [00:00<?, ?it/s]

  0%|          | 0/173504 [00:00<?, ?it/s]


=== FINAL EVALUATION METRICS ===
 MAP@500:  0.0088
 NDCG@500: 0.0482

Saving latest model to local disk...

 RUN COMPLETE
Final Results -> MAP@500: 0.0088 | NDCG@500: 0.0482
Baseline -> MAP@500:  0.000004 | NDCG@500: 0.000111


In [19]:
# Load Model Artifacts:
print("Loading model artifacts from disk...")
als_model = joblib.load('als_model_dev.joblib')
uri_to_id = joblib.load('uri_to_id.joblib')
id_to_uri = joblib.load('id_to_uri.joblib')
user_items = sparse.load_npz('user_items.npz')
print("Model restored successfully!")


Loading model artifacts from disk...
Model restored successfully!


In [21]:
# MODULE 5: INFERENCE & RECOMMENDATION

# Function Definition 
def recommend_playlists_for_track(test_uri, model, user_items_matrix, uri_to_id,track_meta_path, playlist_meta_path, min_followers=10, n_candidates=5000, n_recs=10):
    if test_uri not in uri_to_id:
        return "Error: Track URI not found in training data."

     # --- Fetch Track Data ---
    track_meta = pd.read_parquet(track_meta_path, filters=[('track_uri', '==', test_uri)])
    track_name = track_meta['track_name'].iloc[0] if not track_meta.empty else "Unknown Track"
    artist_name = track_meta['artist_name'].iloc[0] if not track_meta.empty else "Unknown Artist"
    
    print(f"--- FINDING PLAYLIST FIT FOR: '{track_name}' by {artist_name} ---")
    track_id = uri_to_id[test_uri]


    # Rank Suggestions by ALS Dot Product
    # The implicit library returns the matrix column IDs, which ARE the Spotify PIDs.

    playlist_ids, als_scores = model.recommend(
        userid=track_id, 
        user_items=user_items_matrix[track_id], 
        N=n_candidates
    )
    
    candidates_df = pd.DataFrame({
        'pid': playlist_ids,  # Direct assignment, no dictionary mapping
        'ALS_Confidence': als_scores
    })
    
    # 2. FILTERING by n_followers for Playlists
    meta_df = pd.read_parquet(playlist_meta_path)
    merged_df = candidates_df.merge(meta_df, on='pid', how='inner')
    filtered_df = merged_df[merged_df['num_followers'] >= min_followers]
    
    # Grab the top N
    final_recs = filtered_df.head(n_recs).copy()
    
    # 3. Calculate Cosine Similarity for UI
    track_vector = model.user_factors[track_id]
    track_norm = np.linalg.norm(track_vector)
    
    cosine_sims = []
    for pid in final_recs['pid']:
        playlist_vector = model.item_factors[pid]
        dot_product = final_recs.loc[final_recs['pid'] == pid, 'ALS_Confidence'].values[0]
        playlist_norm = np.linalg.norm(playlist_vector)
        
        if track_norm == 0 or playlist_norm == 0:
            cosine_sims.append(0)
        else:
            cosine_sim = dot_product / (track_norm * playlist_norm)
            cosine_sims.append(min(cosine_sim, 1.0))
            
    # Format cleanly
    final_recs['Cosine_Similarity'] = [f"{round(c * 100, 1)}%" for c in cosine_sims]
    final_recs['ALS_Confidence'] = final_recs['ALS_Confidence'].round(2)
    
    final_recs.insert(0, 'Rank', range(1, len(final_recs) + 1))
    
    return final_recs[['Rank', 'pid', 'name', 'num_followers', 'ALS_Confidence', 'Cosine_Similarity']]

In [23]:

# --- RUN THE TEST --


# Load Data and Model
user_items = sparse.load_npz('user_items.npz')
als_model = joblib.load('als_model.joblib')
uri_to_id = joblib.load('uri_to_id.joblib')


# Only need the Track URI reverse lookup
id_to_uri = {v: k for k, v in uri_to_id.items()}

# Find a track with strong signal for testing
track_counts = user_items.sum(axis=1).A1  
popular_track_id = np.argmax(track_counts) 
sample_uri = id_to_uri[popular_track_id]

results_df = recommend_playlists_for_track(
    test_uri=sample_uri, 
    model=als_model, 
    user_items_matrix=user_items,
    uri_to_id=uri_to_id,
    track_meta_path=track_meta_path,
    playlist_meta_path=playlist_meta_path,
    min_followers=10, 
    n_candidates=5000, 
    n_recs=10
)

results_df

--- FINDING PLAYLIST FIT FOR: 'HUMBLE.' by Kendrick Lamar ---


,Rank,pid,name,num_followers,ALS_Confidence,Cosine_Similarity
2,1,21123,hyphy,16,1.11,41.2%
310,2,47708,fire🔥🔥,13,0.43,37.9%
374,3,14944,RUN,21,0.40,14.5%
420,4,35656,meh,10,0.38,25.1%
792,5,24674,One Last Time,10,0.27,13.4%
872,6,33319,southside,41,0.26,22.2%
915,7,28180,Popular Music,15,0.25,7.4%
1044,8,30469,Trips,10,0.23,16.9%
1308,9,47970,Chill,30,0.20,7.0%
1557,10,44627,The Lacs,13,0.18,11.3%
